# Phase 2: Generative AI

## 1. Generative AI Model Selection & Setup

### Model Selection

Google Gemini 3.5 Flash was selected as the
generative AI model for our credit card default
prediction system.

The model was selected for its text generation
capabilities, ease of integration with Python,
and suitability for generating executive-level
explanations and risk summaries.

The API connection and text generation were
successfully tested before implementing the
prompt templates.

In [1]:
%pip install google-genai

### API Setup and Secure Key Handling

The Google Gen AI SDK is used to integrate the
Gemini API with Python.

The API key is entered securely at runtime using
the getpass module, rather than hardcoding it
in the notebook.

This approach prevents the API key from being
stored in the source code or exposed in the
GitHub repository.

In [2]:
from google import genai
from getpass import getpass

api_key = getpass("Enter your Gemini API key: ")

client = genai.Client(api_key=api_key)

Enter your Gemini API key: ··········


### API Connection and Model Testing

The API connection was verified by retrieving
the list of available generative AI models.

Gemini 3.5 Flash was then tested using a simple
credit card default risk question to verify
its text generation capabilities.

The model successfully generated a response,
confirming that the API integration is functional.

In [3]:
for model in client.models.list():
    if "generateContent" in (model.supported_actions or []):
        print(model.name)


models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-nano-banana-2.1
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-omni-1.1-flash
models/gemini-omni-1.2-flash
models/gemini-3.5-transcribe
models/gemini-3.6-flash
models/gemini-3.7-flash
models/gemini-3.8-flash
models/gemini-4-argon
models/gemini-4-carbon
models/gemi

In [5]:
MODEL_NAME = "gemini-3.5-flash"
print("Using model:", MODEL_NAME)

response = client.models.generate_content(
    model=MODEL_NAME,
    contents="Explain credit card default risk in two sentences."
)

print(response.text)


Using model: gemini-3.5-flash
Credit card default risk is the likelihood that a borrower will fail to make their required payments and ultimately abandon their debt. This poses a financial threat to card issuers, who offset the potential loss of uncollected money by setting credit limits and charging higher interest rates.


## 2. Prompt Template 1: Executive Credit Risk Briefing

### 2.1 Loading Clustering Results

In [8]:
import json
from pathlib import Path

ROOT = Path.cwd()

if ROOT.name == "Generative_AI":
    ROOT = ROOT.parent

profiles_path = (
    ROOT / "Unsupervised_Learning"
    / "clustering_results"
    / "cluster_profiles.json"
)

with open(profiles_path, "r", encoding="utf-8") as file:
    cluster_profiles = json.load(file)

print(json.dumps(cluster_profiles, indent=2))

{
  "0": {
    "segment_name": "Active On-Time Payers",
    "risk_level": "Low",
    "description": "Clients who use their card regularly, with above-average bills and above-average payment amounts, and almost no late months.",
    "recommended_action": "Retain and grow: eligible for credit-limit increases, rewards and cross-selling. Standard monitoring.",
    "key_traits": "median credit limit NT$150,000; median average bill NT$33,790; 0.2 late months out of 6 on average",
    "clients_train": 15009,
    "share_pct": 62.5,
    "default_rate_train_pct": 13.8,
    "default_rate_test_pct": 13.3,
    "risk_vs_average": 0.62,
    "mean_silhouette": 0.391
  },
  "1": {
    "segment_name": "Low-Activity Cardholders",
    "risk_level": "Medium",
    "description": "Clients who barely use their card (very small bills, many months with no activity) despite having the highest credit limits. They pay most of what they owe, but about 1 in 3 had at least one late month.",
    "recommended_action": 

### 2.2 Prompt Template Design

**Template ID:** PT-01

**Template Name:** Executive Credit Risk Briefing

**Intended Use Case:**
Generate a concise credit risk briefing for bank
managers using the supervised learning prediction
and unsupervised learning customer segment.

**Design Rationale:**
The prompt combines the Random Forest prediction
with the K-Means customer segment profile to
provide a data-driven explanation of credit risk.

A structured output format is used to improve
clarity, consistency, and interpretability.

The generated briefing supports human
decision-making rather than replacing it.

### 2.3 Prompt Template Implementation

The prompt uses dynamic placeholders to combine
supervised learning predictions with customer
segment information from unsupervised learning.

The output follows a predefined structure to
ensure clarity and consistency across test cases.

In [9]:
PROMPT_TEMPLATE_1 = """
You are a credit risk analyst assisting bank managers.

Analyze the following machine learning results.

Supervised Learning Results:
- Default Prediction: {prediction}
- Default Probability: {default_probability}%

Unsupervised Learning Results:
- Customer Segment: {segment_name}
- Segment Risk Level: {risk_level}
- Segment Description: {description}
- Key Segment Traits: {key_traits}
- Segment Default Rate: {segment_default_rate}%
- Suggested Segment Action: {recommended_action}

Generate an executive credit risk briefing using
the following structure:

1. Risk Summary
2. Key Risk Factors
3. Customer Segment Interpretation
4. Recommended Actions

Instructions:
- Use clear and professional business language.
- Base the analysis only on the provided information.
- Do not invent facts or numerical values.
- Distinguish individual prediction probability
  from the historical segment default rate.
- Do not assume that segment traits describe
  the individual customer.
- Provide recommendations for human review,
  not final lending decisions.
"""

### 2.4 Dynamic Prompt Generation

A reusable function is implemented to populate
the prompt placeholders with the Random Forest
prediction and the corresponding K-Means
customer segment profile.

In [10]:
def build_prompt_1(prediction, default_probability, cluster_id):

    profile = cluster_profiles[str(cluster_id)]

    return PROMPT_TEMPLATE_1.format(
        prediction=prediction,
        default_probability=round(default_probability * 100, 2),
        segment_name=profile["segment_name"],
        risk_level=profile["risk_level"],
        description=profile["description"],
        key_traits=profile["key_traits"],
        segment_default_rate=profile["default_rate_train_pct"],
        recommended_action=profile["recommended_action"]
    )

### 2.5 Loading Customer Data

The clustered dataset is loaded to prepare
real test cases for Prompt Template 1.

Customer records will be used to obtain
supervised learning predictions and their
corresponding customer segments.

In [11]:
import pandas as pd

data_path = ROOT / "Dataset" / "clustered_data.csv"

clustered_data = pd.read_csv(data_path)

print("Dataset shape:", clustered_data.shape)

print("\nColumn names:")
print(clustered_data.columns.tolist())

clustered_data.head()

Dataset shape: (30000, 37)

Column names:
['ID', 'EDUCATION_0', 'EDUCATION_1', 'EDUCATION_2', 'EDUCATION_3', 'EDUCATION_4', 'EDUCATION_5', 'EDUCATION_6', 'LIMIT_BAL', 'PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6', 'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6', 'AGE', 'MONTHS_LATE', 'UTILIZATION', 'PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6', 'default.payment.next.month', 'split', 'kmeans_cluster', 'gmm_cluster', 'gmm_membership', 'segment_name', 'risk_level']


,ID,EDUCATION_0,EDUCATION_1,EDUCATION_2,EDUCATION_3,EDUCATION_4,EDUCATION_5,EDUCATION_6,LIMIT_BAL,PAY_AMT1,...,PAY_4,PAY_5,PAY_6,default.payment.next.month,split,kmeans_cluster,gmm_cluster,gmm_membership,segment_name,risk_level
0,22789,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.340007,0.918339,...,1.911811,0.240260,0.256087,0,train,2,2,1.000000,Chronic Late Payers,High
1,29007,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.271348,-0.816245,...,-1.530826,-1.537252,-1.489963,0,train,1,1,1.000000,Low-Activity Cardholders,Medium
2,16951,0.0,0.0,1.0,0.0,0.0,0.0,0.0,-2.609497,-1.707251,...,0.190493,0.240260,0.256087,0,train,2,2,0.999998,Chronic Late Payers,High
3,22281,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.678790,0.764288,...,0.190493,0.240260,0.256087,1,train,0,0,1.000000,Active On-Time Payers,Low
4,11347,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.043629,-2.045917,...,0.190493,0.240260,-1.489963,0,train,1,1,1.000000,Low-Activity Cardholders,Medium


### 2.6 Selecting Test Cases

Three customer records are selected from the
test dataset, representing low, medium, and
high-risk customer segments.

This selection allows the prompt to be evaluated
across different customer risk profiles.

In [12]:
test_data = clustered_data[
    clustered_data["split"] == "test"
]

test_cases = (
    test_data.groupby("kmeans_cluster")
    .sample(n=1, random_state=42)
    .sort_values("kmeans_cluster")
)

test_cases[
    ["ID", "kmeans_cluster", "segment_name", "risk_level"]
]

,ID,kmeans_cluster,segment_name,risk_level
26078,18124,0,Active On-Time Payers,Low
24394,9727,1,Low-Activity Cardholders,Medium
24029,24891,2,Chronic Late Payers,High


### 2.7 Loading the Random Forest Model

The trained Random Forest model from Phase 1
will be reused to obtain default predictions
and estimated default probabilities for the
three selected test cases.

In [13]:
model_files = list(
    (ROOT / "Supervised_Learning").rglob("*.joblib")
)

for file in model_files:
    print(file.relative_to(ROOT))

Supervised_Learning/models/rf_model.joblib
Supervised_Learning/models/svm_model.joblib


In [14]:
import joblib

rf_path = (
    ROOT / "Supervised_Learning"
    / "models"
    / "rf_model.joblib"
)

rf_model = joblib.load(rf_path)

print("Random Forest loaded successfully!")
print("Model type:", type(rf_model))
print("Classes:", getattr(rf_model, "classes_", None))
print("Expected features:",
      getattr(rf_model, "n_features_in_", None))
print("Feature names:",
      getattr(rf_model, "feature_names_in_", None))

Random Forest loaded successfully!
Model type: <class 'sklearn.ensemble._forest.RandomForestClassifier'>
Classes: [0 1]
Expected features: 29
Feature names: ['EDUCATION_0' 'EDUCATION_1' 'EDUCATION_2' 'EDUCATION_3' 'EDUCATION_4'
 'EDUCATION_5' 'EDUCATION_6' 'LIMIT_BAL' 'PAY_AMT1' 'PAY_AMT2' 'PAY_AMT3'
 'PAY_AMT4' 'PAY_AMT5' 'PAY_AMT6' 'BILL_AMT1' 'BILL_AMT2' 'BILL_AMT3'
 'BILL_AMT4' 'BILL_AMT5' 'BILL_AMT6' 'AGE' 'MONTHS_LATE' 'UTILIZATION'
 'PAY_0' 'PAY_2' 'PAY_3' 'PAY_4' 'PAY_5' 'PAY_6']


### 2.8 Preparing Model Inputs

The selected customer records are checked
against the original preprocessed dataset
and the Random Forest feature requirements
to ensure consistent model inputs.

In [15]:
preprocessed_data = pd.read_csv(
    ROOT / "Dataset" / "preprocessed_data.csv"
)

required_features = list(rf_model.feature_names_in_)

missing_features = [
    feature for feature in required_features
    if feature not in preprocessed_data.columns
]

print("Required features:", len(required_features))
print("Missing features:", missing_features)
print("Dataset shape:", preprocessed_data.shape)
print("Columns:", preprocessed_data.columns.tolist())

Required features: 29
Missing features: []
Dataset shape: (30000, 32)
Columns: ['ID', 'EDUCATION_0', 'EDUCATION_1', 'EDUCATION_2', 'EDUCATION_3', 'EDUCATION_4', 'EDUCATION_5', 'EDUCATION_6', 'LIMIT_BAL', 'PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6', 'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6', 'AGE', 'MONTHS_LATE', 'UTILIZATION', 'PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6', 'default.payment.next.month', 'split']


In [16]:
selected_customers = (
    preprocessed_data.set_index("ID")
    .loc[test_cases["ID"].tolist()]
)

X_cases = selected_customers[required_features]

print("Input shape:", X_cases.shape)
print("Missing values:", X_cases.isna().sum().sum())

X_cases

Input shape: (3, 29)
Missing values: 0


,EDUCATION_0,EDUCATION_1,EDUCATION_2,EDUCATION_3,EDUCATION_4,EDUCATION_5,EDUCATION_6,LIMIT_BAL,PAY_AMT1,PAY_AMT2,...,BILL_AMT6,AGE,MONTHS_LATE,UTILIZATION,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,PAY_6
ID,,,,,,,,,,,,,,,,,,,,,
18124,0.0,0.0,1.0,0.0,0.0,0.0,0.0,-0.703432,0.281540,0.308414,...,0.586785,-0.04705,0.105940,0.373726,0.012576,0.111849,0.139780,0.190493,2.017773,0.256087
9727,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.197951,-0.261063,-0.236421,...,-0.538867,-0.59082,-0.538890,-1.025763,-0.877791,-0.722324,-0.697642,-0.670167,-0.648496,-0.616938
24891,0.0,1.0,0.0,0.0,0.0,0.0,0.0,-0.397388,0.470669,-0.861905,...,0.645364,-0.59082,2.040431,1.299575,0.012576,0.111849,1.814624,1.911811,2.017773,2.002137


In [17]:
phase1_path = ROOT / "Phase1_Supervised_Learning.ipynb"

with open(phase1_path, "r", encoding="utf-8") as file:
    phase1_notebook = json.load(file)

for cell in phase1_notebook["cells"]:
    if cell["cell_type"] == "code":
        source = "".join(cell["source"])

        if any(term in source for term in [
            "rf_model.fit",
            "RandomForestClassifier(",
            "predict_proba",
            "threshold",
            "StandardScaler",
            "preprocessed_data.csv"
        ]):
            print(source)
            print("-" * 50)

# ===== Load the preprocessed data (prepared in Phase1_Data_Exploration) =====
data = pd.read_csv(ROOT / "Dataset" / "preprocessed_data.csv")

# The file already contains the team's train/test split in the "split" column
train = data[data["split"] == "train"]
test = data[data["split"] == "test"]

# ID is only for linking rows back to the raw data; it is not a model input
drop_cols = ["ID", TARGET, "split"]

X_train = train.drop(columns=drop_cols)
y_train = train[TARGET]
X_test = test.drop(columns=drop_cols)
y_test = test[TARGET]
test_ids = test["ID"]  # kept for the misclassification analysis later

print("Training set:", X_train.shape, "| defaults:", y_train.sum())
print("Test set:    ", X_test.shape, "| defaults:", y_test.sum())

# Safety check: stop if the split is different from the team's split
assert X_train.shape == (24000, 29) and X_test.shape == (6000, 29)
assert y_train.sum() == 5309 and y_test.sum() == 1327
print("Split verified: identical to the team's split.")
------------

In [18]:
for cell in phase1_notebook["cells"]:
    if cell["cell_type"] == "code":
        source = "".join(cell["source"])

        for line in source.splitlines():
            if any(term in line for term in [
                "threshold",
                "thr[",
                "rf_model.fit",
                "rf_model.predict",
                "StandardScaler",
                "joblib.dump"
            ]):
                print(line)

y_pred = best_rf.predict(X_test)               # predicted class (threshold 0.5)
    "threshold": 0.5,
# ===== Decision threshold analysis =====
# (the threshold is chosen without looking at the test set)
threshold_table = pd.DataFrame(rows)
display(threshold_table.round(3))
best_threshold = threshold_table.loc[threshold_table["F1"].idxmax(), "Threshold"]
print("Threshold with the best F1 on training folds:", best_threshold)
# Report the test results for the default, the best-F1 and a high-recall threshold
thresholds_to_report = sorted({0.40, float(best_threshold), 0.50})
threshold_compare = pd.DataFrame({
    for t in thresholds_to_report
threshold_compare.round(3)
joblib.dump(best_rf, model_path, compress=3)   # compressed to keep the file small
test_metrics["tuned_threshold"] = float(best_threshold)
joblib.dump(best_svm, MODELS_DIR / "svm_model.joblib")
# how many false alarms would it make? (Illustration only, not used to pick a threshold.)
      f"(probability threshold {thr[i]:.2

### 2.9 Generating Random Forest Predictions

The trained Random Forest model is used to
generate default predictions and estimated
default probabilities for the three selected
test cases, using the original classification
threshold of 0.5 from Phase 1.

In [19]:
predictions = rf_model.predict(X_cases)

probabilities = rf_model.predict_proba(X_cases)[:, 1]

prediction_results = test_cases[
    ["ID", "kmeans_cluster", "segment_name", "risk_level"]
].copy()

prediction_results["prediction"] = predictions

prediction_results["default_probability"] = probabilities

prediction_results

,ID,kmeans_cluster,segment_name,risk_level,prediction,default_probability
26078,18124,0,Active On-Time Payers,Low,1,0.547173
24394,9727,1,Low-Activity Cardholders,Medium,0,0.498034
24029,24891,2,Chronic Late Payers,High,1,0.690066


### 2.10 Testing Prompt Template 1

Prompt Template 1 is evaluated using three
real test cases representing different
customer segments and risk levels.

The Random Forest predictions, estimated
default probabilities, and cluster profiles
are provided as context to the generative
AI model to produce executive-level
explanations.

In [20]:
prompt_1_results = []

for i, (_, row) in enumerate(
    prediction_results.iterrows(), start=1
):
    prediction_label = (
        "Default" if row["prediction"] == 1
        else "No Default"
    )

    prompt = build_prompt_1(
        prediction=prediction_label,
        default_probability=float(row["default_probability"]),
        cluster_id=int(row["kmeans_cluster"])
    )

    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt
    )

    prompt_1_results.append({
        "case": i,
        "customer_id": int(row["ID"]),
        "prompt": prompt,
        "output": response.text
    })

    print(f"Test Case {i}")
    print(response.text)
    print("=" * 70)

Test Case 1
# Executive Credit Risk Briefing

## 1. Risk Summary
The supervised machine learning model predicts a status of **Default** for this customer, with an individual **Default Probability of 54.72%**. 

Conversely, the unsupervised clustering model places this customer into the **"Active On-Time Payers"** segment, a group characterized by a **Low** risk level and a historical **Segment Default Rate of 13.8%**. 

Because the individual's predicted default probability (54.72%) is substantially higher than the historical baseline of their assigned segment (13.8%), this account presents a notable divergence that requires targeted review.

---

## 2. Key Risk Factors
* **High Individual Default Probability:** The supervised model evaluates the customer's specific risk of default at 54.72%, triggering a formal "Default" prediction.
* **Model Divergence:** The individual's elevated risk score conflicts directly with the broader segment's historically low default rate (13.8%). This sug

### 2.11 Saving Prompt Template 1 Outputs

The generated prompts and responses for all
three test cases are saved for documentation,
evaluation, and reproducibility.

In [21]:
output_path = (
    ROOT / "Generative_AI"
    / "prompt_1_results.json"
)

with open(output_path, "w", encoding="utf-8") as file:
    json.dump(
        prompt_1_results,
        file,
        indent=4,
        ensure_ascii=False
    )

print("Number of completed test cases:", len(prompt_1_results))
print("Results saved to:", output_path.relative_to(ROOT))

Number of completed test cases: 3
Results saved to: Generative_AI/prompt_1_results.json


### 2.12 Analysis and Evaluation of Prompt Template 1

Prompt Template 1 was successfully tested using
three real customer cases representing low,
medium, and high-risk customer segments.

#### Case 1: Low-Risk Segment

The model correctly reported a default prediction
with a default probability of 54.72%, despite the
customer belonging to a low-risk segment.

The generated response identified the difference
between individual risk and segment-level risk.
However, it suggested possible individual risk
drivers without sufficient supporting evidence.

#### Case 2: Medium-Risk Segment

The model correctly reported a No Default
prediction with a default probability of 49.80%.

The response recognized that the probability was
close to the classification threshold of 50%
and highlighted the need for further review.

#### Case 3: High-Risk Segment

The model correctly reported a default prediction
with a default probability of 69.01%.

The response identified the alignment between
the individual prediction and the high-risk
customer segment.

#### Strengths

- Accurate reporting of predictions and probabilities.
- Clear and structured executive-level explanations.
- Effective integration of supervised and
  unsupervised learning results.
- Clear distinction between individual predictions
  and segment-level characteristics.

#### Limitations

- Some explanations may include unsupported assumptions.
- Individual risk drivers cannot be confirmed without
  additional customer-level evidence.
- Generated recommendations require human validation.
- Testing only three cases limits generalizability.

#### Conclusion

Prompt Template 1 successfully generated structured
executive credit risk briefings for all three
test cases.

The results demonstrate the potential of generative
AI to explain machine learning outputs in an
understandable format. However, the generated
explanations should support human decision-making
rather than replace it.

## 3. Prompt Template 2: Customer-Level Risk Decision Memo

### 3.1 Prompt Template Design

**Template ID:** PT-02

**Template Name:** Customer-Level Risk Decision Memo

**Intended Use Case:**
A credit officer gets an alert for one customer and needs to decide what to do
with that account. PT-02 produces a short decision memo that says which action
level applies, explains why using **this customer's own repayment history**, and
states what is uncertain. The output is structured JSON, so the dashboard can
show each part in its own place (action badge, evidence list, next steps).

**Design Rationale:**

PT-01 gives the model only the prediction and the segment profile. As noted in
Section 2.12, this means the model **cannot confirm individual risk drivers** and
sometimes guesses them. PT-02 was designed to fix this, and it differs from PT-01
in the following ways:

| Aspect | PT-01 (Executive Briefing) | PT-02 (Decision Memo) |
|---|---|---|
| Customer information | Prediction + segment only | Customer's real 6-month history (status, bills, payments in NT$) + derived indicators |
| Random Forest information in the prompt | prediction + probability only | Prediction + probability and the RF's main drivers from Phase 1 feature importance (PAY_0, MONTHS_LATE, UTILIZATION) |
| Prompting technique | Zero-shot, instruction-based | **One-shot** (a worked example) + an explicit **decision policy** |
| Output format | Structured text (4 fixed headings) | Structured JSON (machine-readable) so the dashboard can use each part directly |
| Grounding rule | "Do not invent facts" | Every evidence item must name the input field it comes from |
| Audience | Senior managers (overview) | Credit officers (account-level action) |

- **Evidence-cited reasoning:** every evidence item has a `source_field`, so a
  reviewer can check each claim against the input. This reduces hallucination
  and makes the memo auditable.
- **Fixed decision policy:** the action level is tied to probability bands. This
  keeps recommendations consistent across customers instead of depending on
  how the model phrases things.
- **Fairness by design:** sex, education, marriage status and age are **not**
  sent to the model. Phase 1 showed they have almost no predictive value, and
  leaving them out stops the memo from justifying decisions with demographics.
- **Real units:** the Random Forest inputs are standardized (z-scores), which a
  language model cannot interpret. PT-02 reads the original NT$ values from
  `raw_kaggle_data.csv` for the same customer IDs.

**Prompt Structure (placeholders):**
`{prediction}`, `{default_probability}`, `{customer_facts}`, `{model_drivers}`,
`{segment_name}`, `{risk_level}`, `{segment_default_rate}`, `{recommended_action}`

**Assumptions & Limitations:**
- Assumes the Random Forest probability is reasonably calibrated. The action
  bands are policy choices made by our team, not bank regulation.
- The global feature importance explains the model in general, not this exact
  prediction (no per-customer SHAP values), so the memo describes evidence that
  is *consistent with* the model's drivers rather than the model's exact reasoning.
- The one-shot example uses a fictional customer and may bias the length and
  style of the outputs.
- JSON output is easy to use in the dashboard but less pleasant to read as raw text.

### 3.2 Preparing Customer-Level Facts

The original (unscaled) records are loaded so the memo can use real amounts.
Repayment status codes are converted into plain words, using the dataset's
coding: -2 = no card use, -1 = paid in full, 0 = paid the minimum (revolving
credit), 1-9 = number of months the payment is late.

Three indicators from Phase 1 are also computed in readable units:
- **Months late**: number of the last 6 months with a late payment (MONTHS_LATE).
- **Utilization**: latest bill / credit limit (UTILIZATION).
- **Payment ratio**: total paid / total billed over the 6 months.

In [22]:
raw_data = pd.read_csv(
    ROOT / "Dataset" / "raw_kaggle_data.csv"
).set_index("ID")

# Column order in the dataset: index 0 = September (most recent) ... 5 = April
MONTHS = ["September", "August", "July", "June", "May", "April"]
PAY_STATUS_COLS = ["PAY_0", "PAY_2", "PAY_3", "PAY_4", "PAY_5", "PAY_6"]
BILL_COLS = [f"BILL_AMT{i}" for i in range(1, 7)]
PAY_AMT_COLS = [f"PAY_AMT{i}" for i in range(1, 7)]


def describe_status(code):
    """Convert a repayment status code into plain words."""
    code = int(code)
    if code == -2:
        return "no card use"
    if code == -1:
        return "paid in full"
    if code == 0:
        return "paid the minimum (revolving credit)"
    return f"{code} month{'s' if code > 1 else ''} late"


def build_customer_facts(customer_id):
    """Build a readable fact sheet for one customer.

    Demographic columns (SEX, EDUCATION, MARRIAGE, AGE) are left out on
    purpose so the memo cannot base decisions on them.
    """
    row = raw_data.loc[customer_id]

    months_late = int((row[PAY_STATUS_COLS] > 0).sum())
    utilization = row["BILL_AMT1"] / row["LIMIT_BAL"] * 100
    total_billed = row[BILL_COLS].clip(lower=0).sum()
    total_paid = row[PAY_AMT_COLS].sum()

    lines = [f"- Credit limit: NT${row['LIMIT_BAL']:,.0f}"]
    lines.append("- Monthly history (most recent first):")
    for month, s, b, p in zip(MONTHS, PAY_STATUS_COLS, BILL_COLS, PAY_AMT_COLS):
        lines.append(
            f"  - {month}: status = {describe_status(row[s])}; "
            f"bill = NT${row[b]:,.0f}; payment = NT${row[p]:,.0f}"
        )

    lines.append(f"- Months late (last 6 months): {months_late} of 6")
    lines.append(f"- Utilization (latest bill / limit): {utilization:.1f}%")
    if total_billed > 0:
        lines.append(
            f"- Payment ratio (paid / billed, 6 months): "
            f"{total_paid / total_billed * 100:.1f}%"
        )
    else:
        lines.append("- Payment ratio: not applicable (nothing billed)")

    return "\n".join(lines)


print(build_customer_facts(int(prediction_results.iloc[0]["ID"])))

- Credit limit: NT$60,000
- Monthly history (most recent first):
  - September: status = paid the minimum (revolving credit); bill = NT$34,604; payment = NT$1,900
  - August: status = paid the minimum (revolving credit); bill = NT$35,625; payment = NT$1,953
  - July: status = paid the minimum (revolving credit); bill = NT$36,701; payment = NT$3,300
  - June: status = paid the minimum (revolving credit); bill = NT$39,048; payment = NT$0
  - May: status = 2 months late; bill = NT$37,269; payment = NT$1,430
  - April: status = paid the minimum (revolving credit); bill = NT$38,000; payment = NT$1,300
- Months late (last 6 months): 1 of 6
- Utilization (latest bill / limit): 57.7%
- Payment ratio (paid / billed, 6 months): 4.5%


### 3.3 Model Driver Summary

This is a short summary of the Random Forest feature importance results from
Phase 1 (Section 7 of `Phase1_Supervised_Learning.ipynb`). It tells the language model
which kinds of evidence the classifier relies on, so the memo focuses on the same
signals instead of drivers the model does not use.

In [23]:
MODEL_DRIVERS = """\
- The two strongest drivers are the most recent repayment status (PAY_0)
  and the number of late months in the last 6 months (MONTHS_LATE).
- Next come utilization (latest bill / credit limit) and the repayment
  status of earlier months (PAY_2, PAY_3).
- Bill and payment amounts have a smaller effect.
- Demographics (age, education) have almost no influence on the model."""

print(MODEL_DRIVERS)

- The two strongest drivers are the most recent repayment status (PAY_0)
  and the number of late months in the last 6 months (MONTHS_LATE).
- Next come utilization (latest bill / credit limit) and the repayment
  status of earlier months (PAY_2, PAY_3).
- Bill and payment amounts have a smaller effect.
- Demographics (age, education) have almost no influence on the model.


### 3.4 Prompt Template Implementation

The template has four parts: (1) role and task, (2) a decision policy that maps
probability to an action level, (3) a one-shot example with a **fictional**
customer showing the expected JSON, and (4) the real case with its placeholders.

Curly braces in the JSON example are doubled (`{{ }}`) so that Python's
`str.format()` keeps them as literal braces.

In [24]:
PROMPT_TEMPLATE_2 = """
You are a senior credit risk officer writing a decision memo about ONE
credit card customer for a colleague who must act on the account today.

DECISION POLICY (choose the action level from the default probability):
- Below 35%      -> "Standard monitoring"
- 35% to 49.99%  -> "Enhanced monitoring"
- 50% to 64.99%  -> "Manual review"
- 65% or higher  -> "Urgent intervention"

RULES:
1. Use ONLY the facts given below. Do not invent numbers, dates or events.
2. Every evidence item must name the input field it comes from in
   "source_field" (for example "September status" or "Utilization").
3. Give 2 to 4 evidence items, ordered from most to least important.
   Prefer evidence related to the model drivers.
4. The segment default rate describes a group of customers, not this
   customer. Never present it as this customer's probability.
5. If the customer's own history and the prediction or segment point in
   different directions, explain this in "uncertainty".
6. If the probability is within 5 percentage points of a policy boundary,
   say so in "uncertainty".
7. Do not mention or infer sex, age, education or marital status.
8. Reply with valid JSON only, no text before or after it.

---------------- EXAMPLE (fictional customer, for format only) ----------------
Input:
- Prediction: No Default | Probability: 22.40%
- Customer facts:
  - Credit limit: NT$200,000
  - Monthly history (most recent first):
    - September: status = paid in full; bill = NT$12,500; payment = NT$12,500
    - August: status = paid in full; bill = NT$9,800; payment = NT$9,800
    - (July to April: paid in full every month)
  - Months late (last 6 months): 0 of 6
  - Utilization (latest bill / limit): 6.2%
  - Payment ratio (paid / billed, 6 months): 100.0%
- Segment: Active On-Time Payers (Low risk, segment default rate 13.8%)

Output:
{{
  "customer_id": "EXAMPLE",
  "action_level": "Standard monitoring",
  "headline": "Low predicted risk (22.4%) supported by a clean 6-month repayment record.",
  "evidence": [
    {{"fact": "Paid in full in September and in every earlier month", "source_field": "Monthly history", "effect": "lowers risk"}},
    {{"fact": "0 of the last 6 months were late", "source_field": "Months late", "effect": "lowers risk"}},
    {{"fact": "Latest bill uses only 6.2% of the limit", "source_field": "Utilization", "effect": "lowers risk"}}
  ],
  "segment_context": "Belongs to a low-risk segment (13.8% historical default rate); the individual record agrees with this.",
  "uncertainty": "No conflicting signals; probability is far from the 35% boundary.",
  "next_steps": ["Keep standard monitoring", "Customer may be considered for retention offers after human review"],
  "human_review_required": false
}}
--------------------------------------------------------------------------------

NOW WRITE THE MEMO FOR THIS CUSTOMER.

Customer ID: {customer_id}

Supervised model (Random Forest):
- Prediction: {prediction}
- Default probability: {default_probability}%

What the model relies on in general:
{model_drivers}

Customer facts:
{customer_facts}

Customer segment (K-Means):
- Segment: {segment_name} ({risk_level} risk)
- Historical segment default rate: {segment_default_rate}%
- Suggested segment action: {recommended_action}

Use the same JSON keys as the example. Set "human_review_required" to true
for "Manual review" and "Urgent intervention", or whenever "uncertainty"
describes a conflict.
"""

### 3.5 Dynamic Prompt Generation

`build_prompt_2()` fills the placeholders for one customer. It uses the same
inputs as `build_prompt_1()` (prediction, probability, cluster) plus the
customer ID, which is used to look up that customer's raw history.

In [25]:
def build_prompt_2(customer_id, prediction, default_probability, cluster_id):

    profile = cluster_profiles[str(cluster_id)]

    return PROMPT_TEMPLATE_2.format(
        customer_id=customer_id,
        prediction=prediction,
        default_probability=round(default_probability * 100, 2),
        model_drivers=MODEL_DRIVERS,
        customer_facts=build_customer_facts(customer_id),
        segment_name=profile["segment_name"],
        risk_level=profile["risk_level"],
        segment_default_rate=profile["default_rate_train_pct"],
        recommended_action=profile["recommended_action"]
    )


# Preview the full prompt for the first test case
first = prediction_results.iloc[0]
print(build_prompt_2(
    customer_id=int(first["ID"]),
    prediction="Default" if first["prediction"] == 1 else "No Default",
    default_probability=float(first["default_probability"]),
    cluster_id=int(first["kmeans_cluster"])
))


You are a senior credit risk officer writing a decision memo about ONE
credit card customer for a colleague who must act on the account today.

DECISION POLICY (choose the action level from the default probability):
- Below 35%      -> "Standard monitoring"
- 35% to 49.99%  -> "Enhanced monitoring"
- 50% to 64.99%  -> "Manual review"
- 65% or higher  -> "Urgent intervention"

RULES:
1. Use ONLY the facts given below. Do not invent numbers, dates or events.
2. Every evidence item must name the input field it comes from in
   "source_field" (for example "September status" or "Utilization").
3. Give 2 to 4 evidence items, ordered from most to least important.
   Prefer evidence related to the model drivers.
4. The segment default rate describes a group of customers, not this
   customer. Never present it as this customer's probability.
5. If the customer's own history and the prediction or segment point in
   different directions, explain this in "uncertainty".
6. If the probability is w

### 3.6 Testing Prompt Template 2

PT-02 is tested on the **same three customers** used for PT-01 (one from each
K-Means segment), with the same Random Forest predictions. Keeping the cases
the same means any difference in the outputs comes from the prompt design.

Gemini's JSON mode (`response_mime_type="application/json"`) is enabled so the
reply can be parsed directly. Temperature is left at the default, the same
as in PT-01, so the comparison stays fair.

In [26]:
from google.genai import types

json_config = types.GenerateContentConfig(
    response_mime_type="application/json"
)

prompt_2_results = []

for i, (_, row) in enumerate(
    prediction_results.iterrows(), start=1
):
    prediction_label = (
        "Default" if row["prediction"] == 1
        else "No Default"
    )

    prompt = build_prompt_2(
        customer_id=int(row["ID"]),
        prediction=prediction_label,
        default_probability=float(row["default_probability"]),
        cluster_id=int(row["kmeans_cluster"])
    )

    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt,
        config=json_config
    )

    # Check that the reply is valid JSON (needed for the dashboard)
    try:
        parsed = json.loads(response.text)
        json_valid = True
    except json.JSONDecodeError:
        parsed = None
        json_valid = False

    prompt_2_results.append({
        "case": i,
        "customer_id": int(row["ID"]),
        "prompt": prompt,
        "output": response.text,
        "parsed_output": parsed,
        "json_valid": json_valid
    })

    print(f"Test Case {i}  (customer {int(row['ID'])}, "
          f"{row['segment_name']})  JSON valid: {json_valid}")
    print(json.dumps(parsed, indent=2, ensure_ascii=False)
          if parsed else response.text)
    print("=" * 70)

Test Case 1  (customer 18124, Active On-Time Payers)  JSON valid: True
{
  "customer_id": "18124",
  "action_level": "Manual review",
  "headline": "Moderate-high predicted default risk (54.72%) driven by past delinquency, revolving balance, and high utilization.",
  "evidence": [
    {
      "fact": "September status was paid the minimum (revolving credit) on a bill of NT$34,604 with a payment of NT$1,900",
      "source_field": "September status",
      "effect": "raises risk"
    },
    {
      "fact": "Recorded 1 of 6 months late in the last 6 months, having been 2 months late in May",
      "source_field": "Months late",
      "effect": "raises risk"
    },
    {
      "fact": "Latest bill uses 57.7% of the NT$60,000 credit limit",
      "source_field": "Utilization",
      "effect": "raises risk"
    },
    {
      "fact": "6-month payment ratio is low at 4.5%",
      "source_field": "Payment ratio",
      "effect": "raises risk"
    }
  ],
  "segment_context": "Belongs to the Ac

### 3.7 Checking Output Quality

Two automatic checks are run on every PT-02 output:

1. **Structure check:** all required keys are present, the action level is
   one of the four allowed values, and it matches the decision policy for
   the given probability.
2. **Number grounding check:** every number in the output must also appear
   in the prompt. A number that does not appear may be a hallucination and
   is flagged for manual review. (Small counts such as "2 evidence items" can
   cause false alarms, so flagged numbers are checked by hand.)

In [27]:
import re

REQUIRED_KEYS = [
    "customer_id", "action_level", "headline", "evidence",
    "segment_context", "uncertainty", "next_steps",
    "human_review_required"
]


def expected_action(probability_pct):
    """Action level that the decision policy requires."""
    if probability_pct < 35:
        return "Standard monitoring"
    if probability_pct < 50:
        return "Enhanced monitoring"
    if probability_pct < 65:
        return "Manual review"
    return "Urgent intervention"


def numbers_in(text):
    """Extract numbers such as 54.72, 60,000 or 13.8 as normalized strings."""
    found = re.findall(r"\d[\d,]*(?:\.\d+)?", text)
    cleaned = set()
    for n in found:
        n = n.replace(",", "").rstrip(".")
        if "." in n:
            n = n.rstrip("0").rstrip(".")
        cleaned.add(n)
    return cleaned


def ungrounded_numbers(output_text, prompt_text):
    """Numbers in the output that never appear in the prompt."""
    return sorted(numbers_in(output_text) - numbers_in(prompt_text))


check_rows = []
for result, (_, row) in zip(prompt_2_results, prediction_results.iterrows()):
    parsed = result["parsed_output"] or {}
    prob_pct = float(row["default_probability"]) * 100

    check_rows.append({
        "case": result["case"],
        "customer_id": result["customer_id"],
        "json_valid": result["json_valid"],
        "missing_keys": [k for k in REQUIRED_KEYS if k not in parsed],
        "action_level": parsed.get("action_level"),
        "policy_action": expected_action(prob_pct),
        "action_matches_policy": parsed.get("action_level") == expected_action(prob_pct),
        "n_evidence": len(parsed.get("evidence", [])),
        "all_evidence_cited": all(
            e.get("source_field") for e in parsed.get("evidence", [])
        ),
        "ungrounded_numbers": ungrounded_numbers(result["output"], result["prompt"])
    })

pt2_checks = pd.DataFrame(check_rows)
pt2_checks

,case,customer_id,json_valid,missing_keys,action_level,policy_action,action_matches_policy,n_evidence,all_evidence_cited,ungrounded_numbers
0,1,18124,True,[],Manual review,Manual review,True,4,True,[]
1,2,9727,True,[],Enhanced monitoring,Enhanced monitoring,True,4,True,[]
2,3,24891,True,[],Urgent intervention,Urgent intervention,True,4,True,[]


### 3.8 Comparing Against the Actual Outcomes

The real outcome (`default.payment.next.month`) was **never** given to the
language model. It is shown here only to judge whether each memo would have
led a credit officer in the right direction, and how it handles a wrong
prediction.

In [28]:
actual_outcomes = (
    preprocessed_data.set_index("ID")
    .loc[prediction_results["ID"], "default.payment.next.month"]
    .values
)

outcome_table = prediction_results[
    ["ID", "segment_name", "prediction", "default_probability"]
].copy()
outcome_table["actual"] = actual_outcomes
outcome_table["rf_correct"] = (
    outcome_table["prediction"] == outcome_table["actual"]
)
outcome_table["pt2_action"] = [
    (r["parsed_output"] or {}).get("action_level") for r in prompt_2_results
]
outcome_table["pt2_human_review"] = [
    (r["parsed_output"] or {}).get("human_review_required")
    for r in prompt_2_results
]

outcome_table

,ID,segment_name,prediction,default_probability,actual,rf_correct,pt2_action,pt2_human_review
26078,18124,Active On-Time Payers,1,0.547173,0,False,Manual review,True
24394,9727,Low-Activity Cardholders,0,0.498034,1,False,Enhanced monitoring,True
24029,24891,Chronic Late Payers,1,0.690066,0,False,Urgent intervention,True


### 3.9 Side-by-Side Examples: PT-01 vs PT-02

The outputs of both templates for the same customer are shown next to each
other. A few simple descriptive numbers are added for each case. The full
quantitative evaluation (readability, keyword alignment, preference
simulation) is done in the evaluation section.

In [29]:
with open(ROOT / "Generative_AI" / "prompt_1_results.json",
          "r", encoding="utf-8") as file:
    prompt_1_saved = json.load(file)

comparison_rows = []
for r1, r2 in zip(prompt_1_saved, prompt_2_results):
    comparison_rows.append({
        "case": r1["case"],
        "customer_id": r1["customer_id"],
        "PT-01 words": len(r1["output"].split()),
        "PT-02 words": len(r2["output"].split()),
        "PT-01 ungrounded numbers": len(
            ungrounded_numbers(r1["output"], r1["prompt"])),
        "PT-02 ungrounded numbers": len(
            ungrounded_numbers(r2["output"], r2["prompt"])),
        "PT-02 machine-readable": r2["json_valid"]
    })

side_by_side = pd.DataFrame(comparison_rows)
display(side_by_side)

for r1, r2 in zip(prompt_1_saved, prompt_2_results):
    print(f"\n{'#' * 30}  CASE {r1['case']} "
          f"(customer {r1['customer_id']})  {'#' * 30}")
    print("\n----- PT-01: Executive Credit Risk Briefing -----\n")
    print(r1["output"])
    print("\n----- PT-02: Customer-Level Risk Decision Memo -----\n")
    print(json.dumps(r2["parsed_output"], indent=2, ensure_ascii=False)
          if r2["parsed_output"] else r2["output"])

,case,customer_id,PT-01 words,PT-02 words,PT-01 ungrounded numbers,PT-02 ungrounded numbers,PT-02 machine-readable
0,1,18124,419,199,0,0,True
1,2,9727,414,183,0,0,True
2,3,24891,355,185,0,0,True



##############################  CASE 1 (customer 18124)  ##############################

----- PT-01: Executive Credit Risk Briefing -----

# Executive Credit Risk Briefing

## 1. Risk Summary
The supervised machine learning model predicts a status of **Default** for this customer, with an individual **Default Probability of 54.72%**. 

Conversely, the unsupervised clustering model places this customer into the **"Active On-Time Payers"** segment, a group characterized by a **Low** risk level and a historical **Segment Default Rate of 13.8%**. 

Because the individual's predicted default probability (54.72%) is substantially higher than the historical baseline of their assigned segment (13.8%), this account presents a notable divergence that requires targeted review.

---

## 2. Key Risk Factors
* **High Individual Default Probability:** The supervised model evaluates the customer's specific risk of default at 54.72%, triggering a formal "Default" prediction.
* **Model Divergence:** T

### 3.10 Saving Prompt Template 2 Outputs and Prompt Files

The following are saved for documentation, evaluation and the dashboard:
- `Generative_AI/prompt_2_results.json`: prompts and responses for all three cases.
- `Generative_AI/prompts/`: both prompt templates as text files.
- `Generative_AI/example_outputs/`: example outputs of both templates for each case.

In [30]:
GENAI_DIR = ROOT / "Generative_AI"

# 1. Full PT-02 results
with open(GENAI_DIR / "prompt_2_results.json", "w", encoding="utf-8") as file:
    json.dump(prompt_2_results, file, indent=4, ensure_ascii=False)

# 2. Prompt templates
prompts_dir = GENAI_DIR / "prompts"
prompts_dir.mkdir(exist_ok=True)
(prompts_dir / "PT-01_executive_credit_risk_briefing.txt").write_text(
    PROMPT_TEMPLATE_1.strip(), encoding="utf-8")
(prompts_dir / "PT-02_customer_risk_decision_memo.txt").write_text(
    PROMPT_TEMPLATE_2.strip(), encoding="utf-8")

# 3. Example outputs for both templates
examples_dir = GENAI_DIR / "example_outputs"
examples_dir.mkdir(exist_ok=True)
for r1, r2 in zip(prompt_1_saved, prompt_2_results):
    (examples_dir / f"PT-01_case{r1['case']}_customer{r1['customer_id']}.md"
     ).write_text(r1["output"], encoding="utf-8")
    (examples_dir / f"PT-02_case{r2['case']}_customer{r2['customer_id']}.json"
     ).write_text(
        json.dumps(r2["parsed_output"], indent=2, ensure_ascii=False)
        if r2["parsed_output"] else r2["output"],
        encoding="utf-8")

print("Saved:")
for path in [GENAI_DIR / "prompt_2_results.json",
             *sorted(prompts_dir.iterdir()),
             *sorted(examples_dir.iterdir())]:
    print(" ", path.relative_to(ROOT))

Saved:
  Generative_AI/prompt_2_results.json
  Generative_AI/prompts/PT-01_executive_credit_risk_briefing.txt
  Generative_AI/prompts/PT-02_customer_risk_decision_memo.txt
  Generative_AI/example_outputs/PT-01_case1_customer18124.md
  Generative_AI/example_outputs/PT-01_case2_customer9727.md
  Generative_AI/example_outputs/PT-01_case3_customer24891.md
  Generative_AI/example_outputs/PT-02_case1_customer18124.json
  Generative_AI/example_outputs/PT-02_case2_customer9727.json
  Generative_AI/example_outputs/PT-02_case3_customer24891.json


### 3.11 Analysis and Evaluation of Prompt Template 2

PT-02 was tested on the same three customers as PT-01. All three outputs were
valid JSON with every required key, and the action level matched the decision
policy in **3 of 3** cases. Every evidence item named its source field, and the
number grounding check found **no numbers that were not in the prompt** (Section 3.7).

#### Case 1: Active On-Time Payers (Low-risk segment), customer 18124
RF: Default, 54.72% → policy: **Manual review**. Actual outcome: **no default**.

- **Action:** the memo chose *Manual review*, as required by the policy.
- **Personalization:** unlike PT-01, which could only say that "specific
  individual features or recent behavioral changes" might explain the high
  probability, PT-02 named the customer's actual signals: the 2-month delay in
  May, 57.7% utilization and a 4.5% payment ratio. All values are correct.
- **Uncertainty:** it correctly flagged both the conflict with the low-risk
  segment (13.8%) and that 54.72% is within 5 points of the 50% boundary.
- **Weakness:** all four evidence items were labelled *"raises risk"*. The
  September status (the model's strongest driver) was "paid the minimum", which
  means the customer was **not late** that month. A balanced memo would have
  listed it as a mitigating factor, as well as the 5 of 6 months without a
  delay. The memo leaned towards confirming the prediction, which in this case
  turned out to be wrong.

#### Case 2: Low-Activity Cardholders (Medium-risk segment), customer 9727
RF: No Default, 49.80% → policy: **Enhanced monitoring**. Actual outcome: **default**.

- **Action:** *Enhanced monitoring*, as required by the policy, and it flagged
  that 49.8% is right at the 50% Manual review boundary.
- **Accuracy:** the evidence (paid in full every month, 0 late months, 0.2%
  utilization, 100% payment ratio) is correct and fairly describes the record.
- **Safety:** the memo did not try to explain the 49.8% probability with
  reasons it did not have. It stated the conflict between the clean history and
  the high probability, set `human_review_required` to true, and kept the
  segment advice to watch for spending spikes.
- **Weakness:** every evidence item points to *lower* risk, and the headline
  calls the record "flawless". A reader could be reassured by this, yet the
  customer did default. The memo also did not notice that the bill and payment
  were the same small amount (NT$326) every month, a pattern that may deserve a
  closer look but that PT-02 had no instruction to comment on.

#### Case 3: Chronic Late Payers (High-risk segment), customer 24891
RF: Default, 69.01% → policy: **Urgent intervention**. Actual outcome: **no default**.

- **Action:** *Urgent intervention*, as required by the policy.
- **Personalization:** the evidence used the customer's own data: 4 of 6
  months late (April to July, each 2 months late), 95.7% utilization and a
  24.3% payment ratio. All values were checked against the raw data and are correct.
- **Strongest result:** in `uncertainty`, the memo noticed that the customer
  **improved recently**: a large NT$65,007 payment in July and no delay in
  August and September. This is exactly the kind of signal that could explain
  why the customer did not default. PT-01 only described the segment and did not see it.
- **Weakness:** the next steps copy the segment's suggested action almost word
  for word ("freeze limit increases, contact for repayment plans, prioritize in
  collections"), even though the memo itself found signs of recovery.

#### Comparison with PT-01 (same cases)

| | PT-01 | PT-02 |
|---|---|---|
| Words per output | 355-419 | 183-199 (about half) |
| Customer-specific risk drivers | No (only prediction and segment) | Yes, in all 3 cases |
| Ungrounded numbers | 0 | 0 |
| Consistent action level | Not defined | Matches policy in 3/3 |
| Recent improvement in Case 3 noticed | No | Yes |
| Machine-readable for the dashboard | No | Yes (JSON) |

Both templates avoided invented numbers and kept the individual probability
separate from the segment default rate. PT-01 reads better as a report for a
manager. PT-02 is shorter, more specific to the customer, and easier to
check. A full quantitative comparison (readability, keyword alignment,
preference simulation) is done in the evaluation section.

#### Strengths
- Evidence is specific to the customer and every claim can be traced back to
  an input field. No unsupported numbers were found.
- The decision policy made the action level consistent and predictable (3/3).
- The `uncertainty` field was useful in every case: it flagged boundary cases,
  conflicts with the segment, and the recent improvement in Case 3.
- The JSON output can be used directly by the dashboard without any parsing of free text.
- About half the length of PT-01, which suits a credit officer who must act quickly.

#### Limitations
- **The memo can only be as good as the classifier.** Section 3.8 shows that
  the Random Forest was wrong for all three customers, and PT-02 still built a
  confident explanation around each prediction. The `uncertainty` field and
  `human_review_required` reduce this risk but do not remove it.
- **One-sided evidence:** in Cases 1 and 2, all evidence items pointed in one
  direction even when the data contained signals for the other side. A rule
  asking for at least one counter-signal would make the memos more balanced.
- **`human_review_required` was true in all three cases.** Because each case was
  near a boundary or had a conflict, this follows the rules, but a field that
  is always true does not help a reader prioritize.
- **Rule 6 (boundary warning) applied to every case**, because all three
  probabilities happened to be within 5 points of a boundary. The template was
  not tested on a clear-cut case.
- Next steps sometimes repeat the segment action instead of adapting it to the
  customer (Case 3).
- Raw JSON is harder for a non-expert to read than PT-01's headed text. It
  relies on the dashboard to present it well.
- Only three test cases were used, so these results cannot be generalized.

#### Conclusion

PT-02 achieved its design goal: given the customer's own history, the language
model explained the risk using real, correct and traceable evidence instead
of guessing, in about half the words of PT-01. The decision policy and JSON
format made the output consistent and ready for the dashboard. Its main
weakness is that it tends to support the classifier's prediction, so it must
be used as a support for human review and not as a final decision. Adding a
rule that requires at least one counter-signal is the most useful improvement
for a next version.